# 支持向量机 Support Vector Machines， SVM

本章只讲序列最小化算法

支持向量机核心思想是在特征空间中构建一个最优决策边界（称为超平面），使得不同类别之间的间隔（margin）最大化。

## 寻找最大间隔
分隔超平面的形式可以写成$w^T x+b$，计算点A到分隔超平面的距离就必须给出点到分隔面的法线或者垂线的长度，该值为
$$\frac{\mid w^TA+b\mid}{\mid\mid w \mid\mid}$$
当计算数据点到分隔面的距离并确定分隔面的放置位置时，间隔通过$label * (w^Tx+b)$来计算。这里的label取±1。数据点位于正方向，则$label * (w^Tx+b)$为正数，数据点位于负方向$label * (w^Tx+b)$也为正数，是一个很好的间隔度量  

找到一组参数w，b，使得所有样本点中，离分隔面最近的那个点的“几何间隔”最大  
$$argmax_{w，b} \{ min_n(label*(w^Tx+b)*\frac{1}{\mid\mid w \mid\mid}) \}$$
- $n$：遍历所有样本点。
- $\text{label} \cdot (w^T x + b)$：**函数间隔**。
- 除以 $\|w\|$，就变成了**几何间隔**（真实的点到平面的垂直距离）。
- **$\min_n$** 表示：在所有样本点中，挑出那个**最小的几何间隔**（即**离分隔面最近的那个点**的距离）。
  - 这个最小值就是分类器的**间隔**。
- 对于每一组w，b，计算出当前的“最小间隔”。
- 然后选择让这个最小间隔最大的那一组w，b。

# SVM 原始目标到约束优化问题的转化

## 1. SVM 的原始目标

我们要找到一组参数 $w$ 和 $b$，使得所有样本点中，离分隔超平面最近的那个点（即支持向量）到超平面的几何距离最大。这个目标写为：

$$
\arg\max_{w,b} \left\{ \min_{n=1}^m \left( y_n (w^T x_n + b) \right) \cdot \frac{1}{\|w\|} \right\}
$$

其中：
- $y_n \in \{+1, -1\}$ 是样本 $x_n$ 的类别标签。
- $w^T x_n + b$ 是点 $x_n$ 到超平面的函数值（带符号，未归一化）。
- 乘以 $y_n$ 后，若分类正确则乘积为正，错误则为负。
- 除以 $\|w\|$ 后得到真实的几何距离。
- 最内层的 $\min$ 在所有样本上取最小几何距离。
- 外层的 $\arg\max$ 寻找使这个最小距离最大的 $w$ 和 $b$。

这个形式很难直接优化，因为内部有 $\min$，外部有 $\max$，且 $w$ 出现在分母。

## 2. 转化为等价的约束优化问题

### 2.1 关键观察：几何间隔与函数间隔的关系

- **函数间隔**（未归一化）：$\hat{\gamma}_n = y_n (w^T x_n + b)$
- **几何间隔**：$\gamma_n = \frac{\hat{\gamma}_n}{\|w\|}$

对于同一个超平面 $(w, b)$，如果将 $w$ 和 $b$ 同时乘以一个正数 $k$，得到 $(kw, kb)$：
- 函数间隔 $\hat{\gamma}_n$ 也乘以 $k$。
- 几何间隔 $\gamma_n$ 保持不变（因为分母 $\|kw\| = k\|w\|$）。

**结论**：几何间隔只依赖于超平面的方向和位置，与 $(w,b)$ 的具体大小无关。我们可以利用这个性质，**人为固定函数间隔的最小值**，来简化问题。

### 2.2 固定最小函数间隔为 1

假设我们已经找到了一个最优超平面，其最小几何间隔为 $\gamma^*$。我们可以对 $(w, b)$ 进行缩放，使得离超平面最近的那些点（即支持向量）的函数间隔恰好等于 1。也就是说：

$$
\min_n \hat{\gamma}_n = 1 \quad \Longrightarrow \quad y_n (w^T x_n + b) \ge 1, \quad \forall n
$$

并且等号至少对一个点成立。此时，几何间隔变为：

$$
\gamma_{\text{geom}} = \frac{\min_n \hat{\gamma}_n}{\|w\|} = \frac{1}{\|w\|}
$$

### 2.3 最大化几何间隔等价于最小化 $\|w\|$

原始目标是最大化 $\gamma_{\text{geom}} = \frac{1}{\|w\|}$。  
由于 $\frac{1}{\|w\|}$ 随 $\|w\|$ 增大而减小，**最大化 $\frac{1}{\|w\|}$ 等价于最小化 $\|w\|$**。  
为了数学上方便（求导时消去根号），通常最小化 $\frac{1}{2} \|w\|^2$，这等价于最小化 $\|w\|$。

### 2.4 得到带约束的优化问题

结合上述缩放固定（最小函数间隔为 1），我们得到如下**凸二次规划**：

$$
\min_{w,b} \; \frac{1}{2} \|w\|^2 \quad \text{subject to} \quad y_n (w^T x_n + b) \ge 1,\; \forall n
$$

- 目标函数是 $w$ 的凸二次函数。
- 约束条件对所有样本 $n$ 是线性的。
- 该问题有唯一全局最优解，且可以使用拉格朗日乘子法进一步转化为对偶形式，从而引入核函数。

## 3. 总结

| 原始问题 | 等价约束优化问题 |
| --- | --- |
| $\max\limits_{w,b} \frac{\min_n y_n(w^T x_n+b)}{\|w\|}$ | $\min\limits_{w,b} \frac{1}{2}\|w\|^2$ |
| 无显式约束（除分类正确外） | s.t. $y_n(w^T x_n+b) \ge 1, \forall n$ |
| 难以直接求解 | 凸二次规划，可求解 |


> **利用“函数间隔可任意缩放而不改变几何间隔”的性质，人为固定最小函数间隔 = 1，从而将原始的“最大化最小几何间隔”问题转化为一个等价的、标准形式的凸二次规划（最小化 $\|w\|^2$ 带线性约束），使求解变得可行。**

1. **原始问题**（约束优化）：
   $$
   \min_{w,b} \frac12 \|w\|^2 \quad \text{s.t.} \quad y_i (w^T x_i + b) \ge 1
   $$

2. 引入拉格朗日乘子 $\alpha_i \ge 0$，构造拉格朗日函数 $L(w,b,\alpha)$。

3. 对 $w$ 和 $b$ 求偏导并令为 0，得到：
   $$
   w = \sum_{i=1}^m \alpha_i y_i x_i, \quad \sum_{i=1}^m \alpha_i y_i = 0
   $$

4. 代回 $L$，消去 $w,b$，就得到上述关于 $\alpha$ 的**对偶问题**，并且要**最大化**这个表达式（加上约束 $\alpha_i \ge 0$ 和 $\sum \alpha_i y_i = 0$）。

$$
 \max_{\alpha}\left[ \quad \sum_{i=1}^{m} \alpha_i \;-\; \frac{1}{2} \sum_{i=1}^{m} \sum_{j=1}^{m} y_i \, y_j \, \alpha_i \, \alpha_j \; \langle \mathbf{x}_i, \mathbf{x}_j \rangle \right]
$$

---
### 1. 原始问题（已转化为约束优化）

$$
\min_{w,b} \frac{1}{2} \|w\|^2 \quad \text{s.t.} \quad y_i (w^T x_i + b) \ge 1,\ i=1..m
$$

### 2. 构造拉格朗日函数

引入拉格朗日乘子 $\alpha_i \ge 0$，每个约束对应一个 $\alpha_i$：

$$
L(w,b,\alpha) = \frac{1}{2} \|w\|^2 - \sum_{i=1}^{m} \alpha_i \left[ y_i (w^T x_i + b) - 1 \right]
$$

### 3. 对 $w$ 和 $b$ 求偏导并令为 0

$$
\frac{\partial L}{\partial w} = w - \sum_{i=1}^{m} \alpha_i y_i x_i = 0 \quad \Rightarrow \quad w = \sum_{i=1}^{m} \alpha_i y_i x_i
$$

$$
\frac{\partial L}{\partial b} = - \sum_{i=1}^{m} \alpha_i y_i = 0 \quad \Rightarrow \quad \sum_{i=1}^{m} \alpha_i y_i = 0
$$

### 4. 代回拉格朗日函数，消去 $w, b$

将 $w$ 的表达式代入 $\frac{1}{2} \|w\|^2$：

$$
\|w\|^2 = \left\langle \sum_{i=1}^{m} \alpha_i y_i x_i,\; \sum_{j=1}^{m} \alpha_j y_j x_j \right\rangle = \sum_{i=1}^{m} \sum_{j=1}^{m} \alpha_i \alpha_j y_i y_j \langle x_i, x_j \rangle
$$

所以：

$$
\frac{1}{2} \|w\|^2 = \frac{1}{2} \sum_{i=1}^{m} \sum_{j=1}^{m} \alpha_i \alpha_j y_i y_j \langle x_i, x_j \rangle
$$

接着处理拉格朗日函数中的求和项：

$$
- \sum_{i=1}^{m} \alpha_i \left[ y_i (w^T x_i + b) - 1 \right] = - \sum_{i=1}^{m} \alpha_i y_i w^T x_i - \sum_{i=1}^{m} \alpha_i y_i b + \sum_{i=1}^{m} \alpha_i
$$

因为 $\sum_{i} \alpha_i y_i = 0$，中间项 $- b \sum_i \alpha_i y_i = 0$。  
而 $\sum_i \alpha_i y_i w^T x_i = w^T \sum_i \alpha_i y_i x_i = w^T w = \|w\|^2$（因为 $w = \sum_i \alpha_i y_i x_i$）。  
因此：

$$
- \sum_{i=1}^{m} \alpha_i y_i w^T x_i = - \|w\|^2
$$

于是拉格朗日函数变为：

$$
L = \frac{1}{2} \|w\|^2 - \|w\|^2 + \sum_{i=1}^{m} \alpha_i = -\frac{1}{2} \|w\|^2 + \sum_{i=1}^{m} \alpha_i
$$

再把 $\|w\|^2$ 展开：

$$
L = \sum_{i=1}^{m} \alpha_i - \frac{1}{2} \sum_{i=1}^{m} \sum_{j=1}^{m} \alpha_i \alpha_j y_i y_j \langle x_i, x_j \rangle
$$

### 5. 得到对偶问题

原问题是 $\min_{w,b} \max_{\alpha \ge 0} L$，在强对偶成立时（SVM 满足），对偶问题是 $\max_{\alpha \ge 0} \min_{w,b} L$。而我们已经通过令偏导为零求出了 $\min_{w,b} L$ 的表达式（它只依赖于 $\alpha$，且正好是上式）。所以对偶问题就是：

$$
\max_{\alpha}\left[ \quad \sum_{i=1}^{m} \alpha_i - \frac{1}{2} \sum_{i=1}^{m} \sum_{j=1}^{m} y_i y_j \alpha_i \alpha_j \langle x_i, x_j\rangle\right]
$$

约束：$\alpha_i \ge 0$，$\sum_i \alpha_i y_i = 0$。